In [ ]:
import sys
from pathlib import Path

# Make the linguaalayam package importable from notebooks/
repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dotenv import load_dotenv
load_dotenv(repo_root / ".env")

In [ ]:
import os
from urllib.parse import quote_plus
import pandas as pd
from sqlalchemy import create_engine, text

def make_engine():
    user = os.environ["DB_USER"]
    password = quote_plus(os.environ["DB_PASSWORD"])
    host = os.environ.get("DB_HOST", "localhost")
    port = os.environ.get("DB_PORT", "5432")
    name = os.environ["DB_NAME"]
    sslmode = os.environ.get("DB_SSLMODE", "")
    sslmode_param = f"?sslmode={sslmode}" if sslmode else ""
    url = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{name}{sslmode_param}"
    return create_engine(url)

engine = make_engine()

with engine.connect() as conn:
    result = conn.execute(text("SELECT version()")).scalar()
    print(result)

## Row counts by source

In [ ]:
pd.read_sql(
    "SELECT source, entry_type, COUNT(*) AS rows FROM dictionary_entries GROUP BY source, entry_type ORDER BY rows DESC",
    engine
)

## Sample entries

In [ ]:
pd.read_sql(
    "SELECT id, source, headword, embed_text FROM dictionary_entries LIMIT 20",
    engine
)